<center> <a href="https://github.com/CyConProject?tab=repositories">
  <img src="https://github.com/CyConProject/Lab/blob/main/Figures/CyCon.png?raw=true" alt="logo" width="80" >
</a>
 </center>

## Implementing Backpropagation

In the previous lab, you examined how a neural network computes a prediction during the **forward pass**. In this lab, you will focus on the learning step: using prediction error to determine how each weight and bias should change.

We will use a simple civil-engineering example: a tiny neural network predicts a **normalized deterioration score** from a condition indicator.
This lab uses a very small **1–1–1 network**: one input, one hidden neuron, and one output neuron. Keeping the network this small makes it easier to see exactly how backpropagation works.

In this lab, you'll:

* **By hand:** compute loss, gradients, and parameter updates for one training step.
* **In code:** implement backpropagation and gradient-descent updates for a tiny network.
* **In practice:** train a small neural network on a structural-health dataset and inspect how the loss changes over time.

> **Why does this matter in civil engineering?** Backpropagation is what allows a model to learn from examples such as strain, deflection, crack width, settlement, or deterioration ratings, so its predictions improve as more inspection or monitoring data become available.

### Objectives

1. Use provided forward-pass results to compute the loss and gradients for a tiny neural network.
2. Apply gradient descent updates to weights and biases.
3. Implement `backward_pass()` and `update_parameters()` in NumPy.
4. Train a simple network and interpret the loss curve in an engineering context.


## 1. Part I – Manual Backpropagation

### 1.1 Scenario and Given Parameters

Suppose a normalized condition indicator from a bridge inspection is

$$
  x = 0.10
$$

and the desired deterioration score is

$$
  T = 0.25.
$$

We will train the following tiny network for **one step**:

* $w_1 = 0.15$
* $b_1 = 0.40$
* $w_2 = 0.45$
* $b_2 = 0.60$
* learning rate $\alpha = 0.40$

To keep the focus on backpropagation, we will **start from the forward-pass results** for this same example instead of recomputing them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# One training example
x = 0.10      # normalized condition indicator
T = 0.25      # desired deterioration score
alpha = 0.40  # learning rate

# Initial parameters
w1 = 0.15
b1 = 0.40
w2 = 0.45
b2 = 0.60

#### 1.1.1 Use the Given Forward-Pass Results

From the same network and input, the forward pass gives:

$$
  z_1 = w_1x + b_1 = 0.415
$$

$$
  a_1 = \sigma(z_1) \approx 0.602286
$$

$$
  z_2 = w_2a_1 + b_2 \approx 0.871029
$$

$$
  a_2 = \sigma(z_2) \approx 0.704960
$$

> **Use these values in the rest of Part I.** The goal here is to practice the backward pass, not to redo the forward-pass calculations from the previous lab.


In [ ]:
z1 = 0.415
a1 = 0.602286
z2 = 0.871029
a2 = 0.704960

print(f"Hidden activation a1 = {a1:.6f}")
print(f"Predicted deterioration score a2 = {a2:.6f}")


#### 1.1.2 Compute the Loss

For this single example, use

$$
  E = \frac{1}{2}(T - a_2)^2
$$


In [ ]:
E = # TODO
print(f"Loss = {E:.6f}")

#### 1.1.3 Compute the Output-Layer Gradients

Use

$$
  \frac{\partial E}{\partial w_2} = (a_2 - T)a_2(1-a_2)a_1
$$

$$
  \frac{\partial E}{\partial b_2} = (a_2 - T)a_2(1-a_2)
$$


In [ ]:
dw2 = # TODO
db2 = # TODO

print(f"dE/dw2 = {dw2:.6f}")
print(f"dE/db2 = {db2:.6f}")

#### 1.1.4 Compute the Hidden-Layer Gradients

Now move one step backward through the hidden neuron:

$$
  \frac{\partial E}{\partial w_1} = (a_2 - T)a_2(1-a_2)w_2a_1(1-a_1)x
$$

$$
  \frac{\partial E}{\partial b_1} = (a_2 - T)a_2(1-a_2)w_2a_1(1-a_1)
$$


In [ ]:
dw1 = # TODO
db1 = # TODO

print(f"dE/dw1 = {dw1:.6f}")
print(f"dE/db1 = {db1:.6f}")

#### 1.1.5 Update the Parameters

Apply one gradient descent step:

$$
  w \leftarrow w - \alpha \frac{\partial E}{\partial w},
  \qquad
  b \leftarrow b - \alpha \frac{\partial E}{\partial b}
$$


In [ ]:
w2_new = # TODO
b2_new = # TODO
w1_new = # TODO
b1_new = # TODO

print(f"Updated w2 = {w2_new:.6f}")
print(f"Updated b2 = {b2_new:.6f}")
print(f"Updated w1 = {w1_new:.6f}")
print(f"Updated b1 = {b1_new:.6f}")

<details>
<summary>Solutions (click to expand)</summary>

```python
E = 0.5 * (T - a2)**2                         # ≈ 0.103494

dw2 = (a2 - T) * a2 * (1 - a2) * a1          # ≈ 0.056993
db2 = (a2 - T) * a2 * (1 - a2)               # ≈ 0.094628

dw1 = (a2 - T) * a2 * (1 - a2) * w2 * a1 * (1 - a1) * x   # ≈ 0.001020
db1 = (a2 - T) * a2 * (1 - a2) * w2 * a1 * (1 - a1)       # ≈ 0.010200

w2_new = w2 - alpha * dw2                    # ≈ 0.427203
b2_new = b2 - alpha * db2                    # ≈ 0.562149
w1_new = w1 - alpha * dw1                    # ≈ 0.149592
b1_new = b1 - alpha * db1                    # ≈ 0.395920
```

</details>

> **Observation:** the output-layer parameters change much more than the hidden-layer parameters in this first step because they are closer to the loss and receive a stronger gradient signal.


## 2. Part II – Helper Functions

To keep the code organized, we will define a few reusable helper functions.

A useful trick is to compute the derivative of sigmoid from the activation value itself:

$$
  \sigma'(z) = \sigma(z)(1-\sigma(z)) = a(1-a)
$$

This avoids re-computing exponentials during backpropagation.


In [ ]:
def sigmoid(z):
    """Sigmoid activation."""
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(a):
    """Derivative of sigmoid, written in terms of activation a."""
    return a * (1.0 - a)


def mse_half(target, prediction):
    """Half-squared error for one example."""
    return 0.5 * (target - prediction)**2

> **Exercise:** verify that `sigmoid(0) = 0.5` and `sigmoid_prime(0.5) = 0.25`.


In [ ]:
# To Do

<details>
<summary>Solutions (click to expand)</summary>

```python
print(sigmoid(0))
print(sigmoid_prime(0.5))
```

> **Answer:** the outputs should be `0.5` and `0.25`.

</details>


## 3. Part III – Implement Backpropagation in Functions

Now we will organize the training step into functions.

* `forward_pass()` is provided for you so the focus stays on backpropagation.
* You will complete `backward_pass()` to compute gradients using the chain rule.
* You will complete `update_parameters()` to apply gradient descent.

In backpropagation, `delta` represents a neuron’s local error signal. We first compute the output-layer error `delta2`, then propagate that error backward through `w2` to get the hidden-layer error `delta1`.

We will represent the network parameters with a dictionary:

```python
params = {'w1': ..., 'b1': ..., 'w2': ..., 'b2': ...}
```


In [ ]:
def forward_pass(x, params):
    """
    One-input, one-hidden-neuron, one-output-neuron network.
    Returns prediction and cached intermediate values.
    """
    z1 = params['w1'] * x + params['b1']
    a1 = sigmoid(z1)

    z2 = params['w2'] * a1 + params['b2']
    a2 = sigmoid(z2)

    cache = {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2}
    return a2, cache


def backward_pass(x, T, params, cache):
    """Compute gradients for one training example."""
    a1 = cache['a1']
    a2 = cache['a2']

    delta2 = # TODO
    dw2 = # TODO
    db2 = # TODO

    delta1 = # TODO
    dw1 = # TODO
    db1 = # TODO

    return {'dw1': dw1, 'db1': db1, 'dw2': dw2, 'db2': db2}


def update_parameters(params, grads, lr):
    """Return a new parameter dictionary after one gradient descent step."""
    updated = params.copy()
    updated['w1'] = # TODO
    updated['b1'] = # TODO
    updated['w2'] = # TODO
    updated['b2'] = # TODO
    return updated


> **Exercise:** complete `backward_pass()` and `update_parameters()`. Then, using the same $x$, $T$, and initial parameters from Part I, run exactly one training step. Your updated values should match your manual work.


In [ ]:
# To Do
params = {'w1': 0.15, 'b1': 0.40, 'w2': 0.45, 'b2': 0.60}


<details>
<summary>Solutions (click to expand)</summary>

```python
def backward_pass(x, T, params, cache):
    a1 = cache['a1']
    a2 = cache['a2']

    delta2 = (a2 - T) * sigmoid_prime(a2)
    dw2 = delta2 * a1
    db2 = delta2

    delta1 = delta2 * params['w2'] * sigmoid_prime(a1)
    dw1 = delta1 * x
    db1 = delta1

    return {'dw1': dw1, 'db1': db1, 'dw2': dw2, 'db2': db2}


def update_parameters(params, grads, lr):
    updated = params.copy()
    updated['w1'] = params['w1'] - lr * grads['dw1']
    updated['b1'] = params['b1'] - lr * grads['db1']
    updated['w2'] = params['w2'] - lr * grads['dw2']
    updated['b2'] = params['b2'] - lr * grads['db2']
    return updated

params = {'w1': 0.15, 'b1': 0.40, 'w2': 0.45, 'b2': 0.60}
prediction, cache = forward_pass(x, params)
loss = mse_half(T, prediction)
grads = backward_pass(x, T, params, cache)
updated_params = update_parameters(params, grads, alpha)

print(f"Prediction = {prediction:.6f}")
print(f"Loss = {loss:.6f}")
print(grads)
print(updated_params)
```

> **Check:** `updated_params['w2']` should be about `0.427203`, and `updated_params['b1']` should be about `0.395920`.

</details>


## 4. Part IV – Train on a Tiny Structural-Health Dataset

Next, we will train the same tiny network on a small synthetic dataset.

Here, each input $x$ is a **normalized deflection indicator** and each target $y$ is a **condition-risk score** between 0 and 1.

The training loop below uses **stochastic gradient descent**, meaning the parameters are updated after each training example rather than once after the full dataset.

> **Note:** this dataset is intentionally small and simplified so the backpropagation steps remain transparent.


In [ ]:
# Tiny structural-health dataset
X_train = np.array([0.05, 0.10, 0.20, 0.35, 0.50, 0.65])
y_train = np.array([0.10, 0.18, 0.30, 0.55, 0.78, 0.90])

print("Inputs:", X_train)
print("Targets:", y_train)

In [ ]:
def train_network(X, y, initial_params, lr=0.4, epochs=4000):
    """Train the tiny network with stochastic gradient descent."""
    params = {k: float(v) for k, v in initial_params.items()}
    history = []

    for epoch in range(epochs):
        total_loss = 0.0

        for x_i, y_i in zip(X, y):
            prediction, cache = forward_pass(x_i, params)
            total_loss += mse_half(y_i, prediction)
            grads = backward_pass(x_i, y_i, params, cache)
            params = update_parameters(params, grads, lr)

        history.append(total_loss / len(X))

    return params, history


initial_params = {'w1': 0.10, 'b1': 0.20, 'w2': -0.30, 'b2': 0.10}
trained_params, history = train_network(X_train, y_train, initial_params, lr=0.4, epochs=4000)

predictions = np.array([forward_pass(x_i, trained_params)[0] for x_i in X_train])

print("Trained parameters:")
for name, value in trained_params.items():
    print(f"  {name} = {value:.6f}")

print("\nPredictions on training data:")
for x_i, y_i, pred_i in zip(X_train, y_train, predictions):
    print(f"x = {x_i:.2f} | target = {y_i:.2f} | prediction = {pred_i:.3f}")

print(f"\nFinal average loss = {history[-1]:.6f}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel('Epoch')
plt.ylabel('Average loss')
plt.title('Training Loss During Backpropagation')
plt.grid(True)
plt.show()

x_grid = np.linspace(X_train.min(), X_train.max(), 200)
y_grid = np.array([forward_pass(x_i, trained_params)[0] for x_i in x_grid])

plt.figure(figsize=(7, 4))
plt.scatter(X_train, y_train, label='Target data')
plt.plot(x_grid, y_grid, label='Network prediction')
plt.xlabel('Normalized deflection indicator')
plt.ylabel('Condition-risk score')
plt.title('Learned Mapping After Training')
plt.legend()
plt.grid(True)
plt.show()

> **Exercise:** compare a small learning rate (`0.05`) with the default learning rate (`0.4`) over `800` epochs. Which one converges faster on this toy dataset?


In [ ]:
# To Do

<details>
<summary>Solutions (click to expand)</summary>

```python
params_a, history_a = train_network(X_train, y_train, initial_params, lr=0.05, epochs=800)
params_b, history_b = train_network(X_train, y_train, initial_params, lr=0.4, epochs=800)

print("Final loss with lr=0.05:", history_a[-1])
print("Final loss with lr=0.4 :", history_b[-1])

plt.figure(figsize=(7, 4))
plt.plot(history_a, label='lr=0.05')
plt.plot(history_b, label='lr=0.4')
plt.xlabel('Epoch')
plt.ylabel('Average loss')
plt.title('Learning Rate Comparison')
plt.legend()
plt.grid(True)
plt.show()
```

> **Answer:** with the provided initialization, `lr = 0.4` reaches a much smaller loss within 800 epochs, while `lr = 0.05` improves only slowly. On this toy problem, the larger step size converges faster.

</details>


## 5. Part V – Experiments and Extensions

Try one or more of the following:

1. **Increase the number of epochs** to 8000. Does the prediction curve change much after 4000 epochs?
2. **Start from different initial weights** and compare the final trained parameters.
3. **Add a second hidden neuron** and extend the backpropagation equations.
4. **Replace the synthetic dataset** with normalized crack-width, strain, or settlement data from a real civil-engineering application.
5. **Track the gradient magnitudes** during training and identify when they become very small.

A low loss on this toy dataset only shows that the learning procedure is working. In real civil-engineering applications, models must also be validated with larger datasets, engineering judgment, and appropriate safety checks.

> **Takeaway:** backpropagation is the step that turns prediction error into learning. In engineering applications, this is what lets a neural network improve its estimates as more monitored or inspected examples are added to the dataset.
